# AutoDL：v6 overlap40 正式训练

适用环境：RTX 5090D、PyTorch 2.8、CUDA 12.8。数据、代码和结果均放在 `/root/autodl-tmp`。

首次运行前，将 Kaggle API 文件上传到 `/root/.kaggle/kaggle.json`。不要在 notebook 中粘贴或输出 token。默认运行 LTL-Net、seed=42、80 epochs；如需跑 DeepLabV3+，只修改训练配置格中的 `MODEL_KIND`。

In [ ]:
from pathlib import Path
import importlib.util
import importlib.metadata
import json
import os
import subprocess
import sys

KAGGLE_DATASET = 'yuanssy/datav6-overlap40'
DATASET_DIRNAME = 'dataset_v6_random811_overlap40'
DATASETS_DIR = Path('/root/autodl-tmp/datasets')
DOWNLOAD_DIR = DATASETS_DIR / 'datav6-overlap40'
DATA_ROOT = DOWNLOAD_DIR / DATASET_DIRNAME
OUTPUT_ROOT = Path('/root/autodl-tmp/outputs')
REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'
REPO_DIR = Path('/root/autodl-tmp/projects/lunar-linear')

for path in (DATASETS_DIR, DOWNLOAD_DIR, OUTPUT_ROOT, REPO_DIR.parent):
    path.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version)
print('数据目标:', DATA_ROOT)
print('结果目录:', OUTPUT_ROOT)
subprocess.run(['nvidia-smi'], check=False)

## 依赖与5090D检查

基础镜像已经包含 PyTorch，下面不会重装 torch/torchvision，只安装缺失的项目依赖。

In [ ]:
required = [
    ('kaggle', 'kaggle'),
    ('rasterio', 'rasterio'),
    ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'),
]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
if (importlib.util.find_spec('segmentation_models_pytorch') is None or
        importlib.metadata.version('segmentation-models-pytorch') != '0.5.0'):
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import torch
import rasterio
import segmentation_models_pytorch as smp

assert torch.cuda.is_available(), 'PyTorch未识别到GPU，请确认实例已用GPU模式开机'
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('Capability:', torch.cuda.get_device_capability(0))
print('SMP:', smp.__version__)
print('Rasterio:', rasterio.__version__)
x = torch.randn(1024, 1024, device='cuda')
print('CUDA计算测试:', (x @ x).mean().item())
del x
torch.cuda.empty_cache()

## 拉取最新代码

仓库不存在时克隆；已存在时只允许 fast-forward 更新，不会强制覆盖服务器上的修改。

In [ ]:
if not REPO_DIR.exists():
    subprocess.check_call([
        'git', 'clone', '--branch', REPO_BRANCH, '--single-branch',
        REPO_URL, str(REPO_DIR)
    ])
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'目录存在但不是Git仓库: {REPO_DIR}')
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])

PROJECT_DIR = REPO_DIR / 'LTL-Net'
assert (PROJECT_DIR / 'scripts/train_ltl.py').is_file(), PROJECT_DIR
commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
print('项目目录:', PROJECT_DIR)
print('Git commit:', commit)

## 从 Kaggle 下载数据

如果目标目录已经存在则跳过下载。若提示凭据错误，请先上传 `/root/.kaggle/kaggle.json` 并在终端执行 `chmod 600 /root/.kaggle/kaggle.json`。

In [ ]:
if DATA_ROOT.is_dir():
    print('数据集已存在，跳过下载:', DATA_ROOT)
else:
    credential_file = Path('/root/.kaggle/kaggle.json')
    credential_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
    assert credential_file.is_file() or credential_env, (
        '缺少Kaggle凭据：请上传 /root/.kaggle/kaggle.json，不要把token粘贴到notebook'
    )
    subprocess.check_call([
        sys.executable, '-m', 'kaggle', 'datasets', 'download',
        '-d', KAGGLE_DATASET, '-p', str(DOWNLOAD_DIR), '--unzip'
    ])

if not DATA_ROOT.is_dir():
    candidates = [p for p in DOWNLOAD_DIR.rglob(DATASET_DIRNAME) if p.is_dir()]
    assert len(candidates) == 1, f'无法唯一定位数据集，发现: {candidates}'
    DATA_ROOT = candidates[0]
print('使用数据集:', DATA_ROOT)

## 数据版本核验

正式训练前核验 1598/200/200、影像-mask配对、5×512×512，以及标签只包含0–4。

In [ ]:
import numpy as np

EXPECTED_TILES = {'train': 1598, 'val': 200, 'test': 200}
EXPECTED_MEAN = np.array([
    0.15665339073973303, 0.6052870962271574, 0.22171011101838023,
    0.5087022443378417, 0.46687463729626205,
])
EXPECTED_STD = np.array([
    0.07239327406001447, 0.35159567816693277, 0.23999408652260576,
    0.18305312443820845, 0.18653673179588806,
])

def tif_files(folder):
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in {'.tif', '.tiff'}])

for split, expected in EXPECTED_TILES.items():
    images = tif_files(DATA_ROOT / split / 'image')
    masks = tif_files(DATA_ROOT / split / 'mask')
    assert len(images) == expected, f'{split}影像数={len(images)}，预期={expected}'
    assert len(masks) == expected, f'{split}掩膜数={len(masks)}，预期={expected}'
    assert {p.stem for p in images} == {p.stem for p in masks}, f'{split}影像-mask不配对'
    mask_by_stem = {p.stem: p for p in masks}
    for index in sorted({0, len(images) // 2, len(images) - 1}):
        with rasterio.open(images[index]) as src:
            image = src.read()
        with rasterio.open(mask_by_stem[images[index].stem]) as src:
            mask = src.read(1)
        assert image.shape == (5, 512, 512), (images[index], image.shape)
        assert mask.shape == (512, 512), (mask_by_stem[images[index].stem], mask.shape)
        assert np.isfinite(image).all(), f'影像存在NaN/Inf: {images[index]}'
        assert set(np.unique(mask)).issubset({0, 1, 2, 3, 4}), np.unique(mask)
    print(f'{split}: {len(images)} 对，抽样检查通过')

stats_path = DATA_ROOT / 'normalization_stats.json'
assert stats_path.is_file(), stats_path
stats = json.loads(stats_path.read_text(encoding='utf-8'))
assert np.allclose(stats['mean'], EXPECTED_MEAN, rtol=0, atol=1e-12), stats['mean']
assert np.allclose(stats['std'], EXPECTED_STD, rtol=0, atol=1e-12), stats['std']
print('Train-only归一化统计:')
print(json.dumps(stats, ensure_ascii=False, indent=2))
print('数据版本核验通过')

## 正式训练配置

默认先跑 LTL-Net。DeepLabV3+ 对照只需把 `MODEL_KIND` 改为 `'baseline'` 并重新运行本格及后续格。`MAX_STEPS=0` 表示每轮完整使用全部数据。若同名结果目录已经存在，代码会停止，防止覆盖权重。

In [ ]:
MODEL_KIND = 'ltl'          # 'ltl' 或 'baseline'
BASELINE_MODEL = 'DeepLabV3Plus'
ENCODER = 'resnet50'
SEED = 42
EPOCHS = 80
MAX_STEPS = 0
NUM_WORKERS = 4
RUN_SUFFIX = 'formal80'

assert MODEL_KIND in {'ltl', 'baseline'}
model_name = 'LTLNet' if MODEL_KIND == 'ltl' else BASELINE_MODEL
run_name = f'v6_overlap40_{model_name}_seed{SEED}_{RUN_SUFFIX}'
result_dir = OUTPUT_ROOT / f'result_{run_name}'
assert not result_dir.exists(), (
    f'结果目录已存在，为防止覆盖已停止：{result_dir}。请检查旧结果或修改RUN_SUFFIX。'
)

common = [
    '--encoder', ENCODER,
    '--data-dir', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
    '--seed', str(SEED),
    '--epochs', str(EPOCHS),
    '--max-steps', str(MAX_STEPS),
    '--num-workers', str(NUM_WORKERS),
    '--run-name', run_name,
]
if MODEL_KIND == 'ltl':
    command = [sys.executable, str(PROJECT_DIR / 'scripts/train_ltl.py'), *common]
else:
    command = [
        sys.executable, str(PROJECT_DIR / 'scripts/train_baseline.py'),
        '--model', BASELINE_MODEL, *common
    ]

print('运行命令:')
print(' '.join(command))
print('输出目录:', result_dir)

In [ ]:
subprocess.check_call(command, cwd=PROJECT_DIR)
print('训练及最终测试完成')

In [ ]:
metrics_path = result_dir / 'metrics.json'
checkpoint_path = result_dir / 'best_model.pth'
assert metrics_path.is_file(), metrics_path
assert checkpoint_path.is_file(), checkpoint_path
result = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(result, ensure_ascii=False, indent=2))
print('\n请下载并保存:')
print(checkpoint_path)
print(metrics_path)